In [46]:
all_result_paths = {
    "RACE": {
        "path" : "week04_Adnan/results/RACE/results.jsonl",
        "sample_count" : 100,
        "base_accuracy" : 0.56,
        "guided_accuracy" : 0.65,
    },
    "SVAMP": {
        "path" : "week04_Adnan/results/SVAMP/results.jsonl",
        "sample_count" : 100,
        "base_accuracy" : 0.55,
        "guided_accuracy" : 0.68,
    },
    "ASDiv": {
        "path" : "week04_Adnan/results/ASDiv/results.jsonl",
        "sample_count" : 100,
        "base_accuracy" : 0.53,
        "guided_accuracy" : 0.64,
    },
    "ARC-Challenge": {
        "path" : "week04_Adnan/results/ARC-Challenge/results.jsonl",
        "sample_count" : 100,
        "base_accuracy" : 0.70,
        "guided_accuracy" : 0.77,
    },
    "CommonsenseQA": {
        "path" : "week04_Adnan/results/COMMONSENSE_QA/results.jsonl",
        "sample_count" : 100,
        "base_accuracy" : 0.68,
        "guided_accuracy" : 0.77,
    },
    'PIQA': {
        "path" : "week04_Adnan/results/PIQA/results.jsonl",
        "sample_count" : 100,
        "base_accuracy" : 0.53,
        "guided_accuracy" : 0.79,
    },

}

In [47]:
import json
from scipy.stats import chi2
def disagreements(results_path):

    records = []
    with open(results_path, "r") as f:
        for line in f:
            single_line = json.loads(line)
            records.append(single_line)

    guided = [r for r in records if r['mode']=='guided']
    baseline = [r for r in records if r['mode']=='baseline']

    b = 0
    c = 0
    for i in range(len(guided)):
        if guided[i]['correct'] == baseline[i]['correct']:
            continue;
        else :
            if guided[i]['correct']: b += 1
            else : c += 1

    return b, c


def chi_p(B, C):
    stat = (abs(B - C) - 1) ** 2 / (B + C)
    p_value   = chi2.sf(stat, 1)

    return stat, p_value


# disagreements('week04_Adnan/results/RACE/results.jsonl')

In [60]:
# Print header
print(f"{'Dataset':<16} {'N':<4} {'Base':<6} {'Guide':<6} {'B':<5} {'C':<5} {'chi2':<7} {'p':<7}")

# Loop through datasets
for dataset in all_result_paths:
    N = all_result_paths[dataset]['sample_count']
    base = all_result_paths[dataset]['base_accuracy'] * 100
    guide = all_result_paths[dataset]['guided_accuracy'] * 100
    b, c = disagreements(all_result_paths[dataset]['path'])
    stat, p_val = chi_p(b, c)

    print(f"{dataset:<15} {N:<5} {base:.1f}%  {guide:.1f}%  {b:<5} {c:<5} {stat:.2f}  {p_val:.2f}")

Dataset          N    Base   Guide  B     C     chi2    p      
RACE            100   56.0%  65.0%  15    6     3.05  0.08
SVAMP           100   55.0%  68.0%  23    10    4.36  0.04
ASDiv           100   53.0%  64.0%  23    12    2.86  0.09
ARC-Challenge   100   70.0%  77.0%  18    11    1.24  0.27
CommonsenseQA   100   68.0%  77.0%  16    7     2.78  0.10
PIQA            100   53.0%  79.0%  37    11    13.02  0.00


In [28]:
from scipy.stats import norm
import math

def mcnemar_power_analysis(p1, p2, alpha=0.05, power=0.80):
    """
    p1 : baseline accuracy (e.g. 0.56)
    p2 : guided accuracy   (e.g. 0.65)
    """
    pi_B   = p2 * (1 - p1)
    pi_C   = p1 * (1 - p2)
    pi_d   = pi_B + pi_C
    signal = pi_B - pi_C

    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)

    N = ((z_alpha * math.sqrt(pi_d) + z_beta * math.sqrt(pi_d - signal**2)) / signal) ** 2
    return math.ceil(N)

# RACE dataset
print(mcnemar_power_analysis(p1=0.56, p2=0.65))

datasets = {
    "SVAMP":         (0.55, 0.68),
    "AQUA-RAT":      (0.40, 0.52),
    "ASDiv":         (0.53, 0.64),
    "ARC-Challenge": (0.70, 0.77),
    "CommonsenseQA": (0.68, 0.77),
    "RACE-High":     (0.56, 0.65),
    "PIQA":          (0.53, 0.79),
}

print(f"{'Dataset':<18} {'Baseline':>10} {'Guided':>8} {'Delta':>7} {'Min N':>7} {'Have':>6} {'Status'}")
print("-" * 65)

for name, (p1, p2) in datasets.items():
    n_min  = mcnemar_power_analysis(p1, p2)
    have   = 100
    status = "✓" if have >= n_min else f"need {n_min - have} more"
    print(f"{name:<18} {p1*100:>9.1f}% {p2*100:>7.1f}% {(p2-p1)*100:>+6.1f}% {n_min:>7} {have:>6}  {status}")

465
Dataset              Baseline   Guided   Delta   Min N   Have Status
-----------------------------------------------------------------
SVAMP                   55.0%    68.0%  +13.0%     222    100  need 122 more
AQUA-RAT                40.0%    52.0%  +12.0%     273    100  need 173 more
ASDiv                   53.0%    64.0%  +11.0%     317    100  need 217 more
ARC-Challenge           70.0%    77.0%   +7.0%     626    100  need 526 more
CommonsenseQA           68.0%    77.0%   +9.0%     388    100  need 288 more
RACE-High               56.0%    65.0%   +9.0%     465    100  need 365 more
PIQA                    53.0%    79.0%  +26.0%      54    100  ✓
